# Constitution Evidence — Kaggle GPU with Hugging Face Transformers

Enable GPU and Internet, and attach the private source dataset. On first run, the setup cell installs dependencies and asks for one Kaggle kernel restart; rerun all cells after restarting. This avoids mixing old and new Hugging Face modules in memory. The notebook checks full-text capacity and saves provisional French responses without building llama.cpp.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, hashlib, time, zipfile, re, shutil
ROOT = Path('/kaggle/working/moroccan-constitution-rag')
INPUT = Path('/kaggle/input')
# Kaggle may mount a ZIP or unpack its files into a dataset directory.
SOURCE_PATH = None  # Set an exact /kaggle/input/... path only if discovery is ambiguous.
paths = [Path(SOURCE_PATH)] if SOURCE_PATH else sorted(INPUT.rglob('*.zip'))
archives = []
for path in paths:
    if not path.is_file():
        continue
    try:
        with zipfile.ZipFile(path) as candidate:
            names = set(candidate.namelist())
            revision = candidate.comment.decode('ascii').strip()
            if {'pyproject.toml', 'notebooks/kaggle_gpu.ipynb'} <= names and re.fullmatch(r'[0-9a-f]{40}', revision):
                archives.append(path)
    except (OSError, zipfile.BadZipFile, UnicodeDecodeError):
        continue
folders = ([Path(SOURCE_PATH)] if SOURCE_PATH and Path(SOURCE_PATH).is_dir() else
           [path.parent for path in INPUT.rglob('pyproject.toml')
            if (path.parent/'notebooks/kaggle_gpu.ipynb').is_file()])
if len(archives) + len(folders) != 1:
    visible = [str(path) for path in sorted(INPUT.rglob('*')) if path.is_file()][:30]
    raise RuntimeError(f'Expected one source ZIP or extracted source directory; '
                       f'found {len(archives)} ZIPs and {len(folders)} directories. '
                       f'Visible input files: {visible}. Set SOURCE_PATH to the correct path if needed.')
if archives:
    with zipfile.ZipFile(archives[0]) as source_zip:
        SOURCE_REVISION = source_zip.comment.decode('ascii').strip()
        if any(Path(name).is_absolute() or '..' in Path(name).parts for name in source_zip.namelist()):
            raise RuntimeError('Source archive contains an unsafe path')
        ROOT.mkdir(parents=True, exist_ok=True)
        source_zip.extractall(ROOT)
    print('Source archive:', archives[0])
else:
    source_dir = folders[0]
    fingerprint = hashlib.sha256()
    for file in sorted(path for path in source_dir.rglob('*') if path.is_file()):
        relative = file.relative_to(source_dir).as_posix()
        fingerprint.update(relative.encode('utf-8') + b'\0')
        fingerprint.update(hashlib.sha256(file.read_bytes()).digest())
    SOURCE_REVISION = 'tree-sha256:' + fingerprint.hexdigest()
    shutil.copytree(source_dir, ROOT, dirs_exist_ok=True)
    print('Source directory:', source_dir)
os.chdir(ROOT)
print('Source revision:', SOURCE_REVISION)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())


In [ ]:
requirements = ['-e','.','huggingface_hub','transformers>=4.51,<5','accelerate','bitsandbytes']
install_marker = ROOT/'.kaggle-transformers-env'
install_signature = hashlib.sha256(json.dumps(requirements).encode()).hexdigest()
if not install_marker.exists() or install_marker.read_text().strip() != install_signature:
    subprocess.run([sys.executable,'-m','pip','install','-q',*requirements],check=True,cwd=ROOT)
    install_marker.write_text(install_signature+'\n')
    raise RuntimeError('Packages installed. Restart the Kaggle session/kernel, then rerun all cells. '
                       'The install cell will skip pip on the next run.')
sys.path.insert(0,str(ROOT/'src'))
import importlib
importlib.invalidate_caches()
import constitution_evidence, bm25s, torch, transformers, bitsandbytes
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator before loading Qwen')
print('Project:',constitution_evidence.__file__)
print('Torch:',torch.__version__,'Transformers:',transformers.__version__,
      'GPU:',torch.cuda.get_device_name(0))
subprocess.run(['apt-get','update','-qq'],check=True,stdout=subprocess.DEVNULL)
subprocess.run(['apt-get','install','-y','-qq','poppler-utils','tesseract-ocr','tesseract-ocr-fra'],check=True,stdout=subprocess.DEVNULL)
(ROOT/'data/raw').mkdir(parents=True,exist_ok=True)
pdf = ROOT/'data/raw/constitution_2011_fr_marocma.pdf'
if not pdf.exists():
    subprocess.run(['curl','-L','--fail','-o',str(pdf),
                    'https://maroc.ma/sites/default/files/2024-10/constitution_du_1er_juillet_2011.pdf'],check=True)
assert hashlib.sha256(pdf.read_bytes()).hexdigest() == '4101d45b4733a6b36eee77bd9ef4175b23160103089b003b13907e8b6d59333d'
from constitution_evidence.ingestion import ingest
from constitution_evidence.store import read_jsonl, write_jsonl, digest
manifest = ingest(ROOT/'configs/corpus.json',root=ROOT,
                  tessdata=Path('/usr/share/tesseract-ocr/5/tessdata'))
passages = read_jsonl(ROOT/'data/processed/passages.jsonl')
questions = read_jsonl(ROOT/'data/annotations/calibration.jsonl')
print('French articles:',manifest['article_count'],'passages:',len(passages),
      'corpus hash:',manifest['corpus_hash'])


In [ ]:
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_ID = 'Qwen/Qwen3-4B'
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
MODEL_SNAPSHOT = snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION,
    allow_patterns=['*.json','*.jinja','*.model','*.txt','*.safetensors'])
TOKEN_LIMIT = 32768  # Qwen3-4B native context; no YaRN extension in this experiment.
MAX_NEW_TOKENS = 768
RESERVE = 128
quantization = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                                  bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_SNAPSHOT,local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_SNAPSHOT,local_files_only=True,
                    quantization_config=quantization,device_map='auto',dtype=torch.float16)
model.eval()
print('Model revision:',MODEL_REVISION,'native token limit:',TOKEN_LIMIT)


In [ ]:
from constitution_evidence.retrieval import BM25
from constitution_evidence.evidence import pack
from constitution_evidence.local_generation import LocalGenerator, RAW_SCHEMA, LOCAL_PROMPT_VERSION
engine = BM25(passages)
message_builder = LocalGenerator(model=MODEL_ID,max_context_chars=200000)
schema_text = json.dumps(RAW_SCHEMA,ensure_ascii=False,sort_keys=True)

def prompt_ids(question,context):
    messages = message_builder.messages(question,context)
    messages[1]['content'] += '\n\nSchéma JSON requis :\n' + schema_text
    ids = tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
                                        enable_thinking=False,return_tensors='pt')
    return ids

def generate_raw(question,context):
    ids = prompt_ids(question,context)
    budget = {'input_tokens':int(ids.shape[-1]),'output_reserve':MAX_NEW_TOKENS+RESERVE,
              'native_context_tokens':TOKEN_LIMIT}
    if budget['input_tokens'] + budget['output_reserve'] > TOKEN_LIMIT:
        return {'budget':budget,'error':{'type':'budget_exceeded',
                                        'message':'Complete prompt plus output reserve exceeds native context'}}
    started = time.monotonic()
    try:
        with torch.inference_mode():
            output = model.generate(ids.to(model.device),max_new_tokens=MAX_NEW_TOKENS,
                                    do_sample=False,pad_token_id=tokenizer.eos_token_id,
                                    use_cache=True)
        generated = output[0,ids.shape[-1]:]
        raw = tokenizer.decode(generated,skip_special_tokens=True)
        result = {'budget':budget,'raw_response':raw,
                  'generated_tokens':int(generated.numel()),
                  'elapsed_seconds':round(time.monotonic()-started,3)}
        try: result['parsed_json'] = json.loads(raw)
        except json.JSONDecodeError: result['json_parse_error'] = True
        return result
    except Exception as exc:
        return {'budget':budget,'error':{'type':type(exc).__name__,'message':str(exc)},
                'elapsed_seconds':round(time.monotonic()-started,3)}


In [ ]:
run_dir = ROOT/'runs/kaggle-transformers-qwen4b'
run_dir.mkdir(parents=True,exist_ok=True)
run_manifest = {'source_revision':SOURCE_REVISION,'model_id':MODEL_ID,
                'model_revision':MODEL_REVISION,'quantization':'bitsandbytes-nf4-4bit-fp16',
                'prompt_version':LOCAL_PROMPT_VERSION+'-hf-schema-v1',
                'prompt_sha256':digest(schema_text+LOCAL_PROMPT_VERSION),
                'corpus_hash':manifest['corpus_hash'],'passages_hash':manifest['passages_hash'],
                'questions_sha256':digest((ROOT/'data/annotations/calibration.jsonl').read_bytes()),
                'token_limit':TOKEN_LIMIT,'max_new_tokens':MAX_NEW_TOKENS,
                'torch_version':torch.__version__,'transformers_version':transformers.__version__,
                'review_status':manifest['review_status'],'status':'provisional_generation_calibration'}
manifest_path = run_dir/'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == run_manifest, 'Run directory belongs to different inputs'
else:
    manifest_path.write_text(json.dumps(run_manifest,ensure_ascii=False,indent=2)+'\n')
fulltext_path = run_dir/'fulltext_smoke.json'
if not fulltext_path.exists():
    result = generate_raw(questions[0]['question'],passages)
    result.update(question_id=questions[0]['question_id'],passage_count=len(passages),
                  corpus_hash=manifest['corpus_hash'],model_revision=MODEL_REVISION)
    fulltext_path.write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
else:
    result = json.loads(fulltext_path.read_text())
print({key:value for key,value in result.items() if key not in {'raw_response','parsed_json'}})
print('Saved:',fulltext_path)


In [ ]:
rows_path = run_dir/'responses.jsonl'
rows = read_jsonl(rows_path) if rows_path.exists() else []
done = {row['question_id'] for row in rows}
assert len(done) == len(rows), 'Duplicate question IDs in saved run'
for q in questions:
    if q['question_id'] in done: continue
    context = pack(engine.search(q['question'],5),7000)
    row = {'question_id':q['question_id'],
           'context_passage_ids':[p['passage_id'] for p in context],
           'context_text_hashes':[p['text_hash'] for p in context]}
    row.update(generate_raw(q['question'],context))
    rows.append(row)
    write_jsonl(rows_path,rows)
    print('Saved',len(rows),'/',len(questions),q['question_id'])
print('Calibration artifact:',rows_path)


The full-text file records whether all 633 French passages fit the native 32,768-token window. A budget failure is an experimental result, not an invitation to truncate the baseline. The calibration file contains raw provisional model responses and exact context IDs. Do not use these outputs as legal judgments or final held-out scores. Return `fulltext_smoke.json` and `manifest.json` for the next decision.
